<a href="https://colab.research.google.com/github/Bvsrkrishna/Data-Analyst-and-Engineering/blob/main/EDA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDA 2

In [ ]:
#@title Setup (run first) { display-mode: "form" }
import sys, os
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False  #@param {type:"boolean"}
if IN_COLAB and USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/clv_project")   # files survive between sessions
elif IN_COLAB:
    BASE = Path("/content/clv_project")                                                                              # lost when runtime resets
else:
    BASE = Path.cwd() / "clv_project"
BASE.mkdir(parents=True, exist_ok=True); os.chdir(BASE)
print("Working folder:", BASE)
REQUIRED = ['data/processed/transactions_clean.csv', 'data/processed/customers_clean.csv', 'data/processed/products_clean.csv']
missing = [p for p in REQUIRED if not (BASE / p).exists()]

Working folder: /content/clv_project


In [ ]:
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
P = BASE / "data" / "processed"; FIG = BASE / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 170, "display.max_columns", 25); sns.set_theme(style="whitegrid")
def h(t): print(f"\n{'='*8} {t} {'='*8}")

tx = pd.read_csv(P / "transactions.csv.csv", parse_date= ["order_id"])
cu = pd.read_csv(P / "customers (1).csv.csv", parse_dates=["signup_date"]); pr = pd.read_csv(P / "products_clean.csv")
df = tx.merge(pr[["product_id", "category"]], on="product_id").merge(cu[["customer_id", "acquisition_channel", "age", "gender", "signup_date"]], on="customer_id")

### 1. Data-quality report function

In [ ]:
h("1. Data-quality report function")
def quality_report(d):
    r = pd.DataFrame({"dtype": d.dtypes.astype(str), "nulls": d.isna().sum(), "null_%": (d.isna().mean() * 100).round(2),
                      "unique": d.nunique(), "sample": d.iloc[0]})
    r["constant"] = r["unique"] <= 1; return r
print(quality_report(df))
assert df.duplicated().sum() == 0 and df.line_revenue.gt(0).all()

### 2. Missing-data strategies (simulate then compare)

In [ ]:
h("2. Missing-data strategies (simulate then compare)")
rng = np.random.default_rng(0); s = df.age.astype(float).copy(); true = s.copy()
mask = rng.random(len(s)) < 0.1; s[mask] = np.nan
cands = {"mean": s.fillna(s.mean()), "median": s.fillna(s.median()),
         "group-median(country)": s.fillna(df.groupby("country").age.transform("median")), "ffill": s.ffill()}
print({k: round(float(np.abs(v[mask] - true[mask]).mean()), 2) for k, v in cands.items()}, "<- MAE vs truth")
df["age_missing_flag"] = s.isna().astype(int)        # missing indicator feature

### 3. Outlier detection

In [ ]:
h("3. Outlier detection")
x = df.groupby("order_id").line_revenue.sum()
q1, q3 = x.quantile([.25, .75]); iqr = q3 - q1; lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
z = (x - x.mean()) / x.std(); med = x.median(); mad = (x - med).abs().median(); mz = 0.6745 * (x - med) / mad
print(f"IQR fences [{lo:.0f}, {hi:.0f}] -> {(x > hi).sum()} | Z>3 -> {(z.abs() > 3).sum()} | MAD>3.5 -> {(mz.abs() > 3.5).sum()}")
xw = x.clip(x.quantile(.01), x.quantile(.99)); print("winsorised max:", round(xw.max(), 1), "| log1p skew:", round(np.log1p(x).skew(), 2), "vs", round(x.skew(), 2))

### 4. Univariate

In [ ]:
h("4. Univariate")
num = df[["quantity", "unit_price", "discount", "line_revenue", "age"]]
print(num.agg(["mean", "median", "std", "skew", "kurt", "min", "max"]).round(2).T)
print(df.order_channel.value_counts(normalize=True).round(3).to_dict(), df.category.nunique(), df.gender.value_counts().to_dict())

### 5. Bivariate

In [ ]:
h("5. Bivariate")
print(df.groupby("acquisition_channel").line_revenue.agg(["mean", "median", "std"]).round(1))
print(pd.crosstab(df.order_channel, df.country, normalize="index").round(2))
print(df.groupby(pd.cut(df.age, [17, 25, 35, 45, 55, 70]), observed=True).line_revenue.mean().round(1).to_dict())

### 6. Correlation (Pearson vs Spearman)

In [ ]:
h("6. Correlation (Pearson vs Spearman)")
print(num.corr().round(2)); print(num.corr(method="spearman").round(2).loc["line_revenue"].to_dict())

### 7. Group stats: transform, rank, cumulative

In [ ]:
h("7. Group stats: transform, rank, cumulative")
orders = df.groupby(["order_id", "customer_id", "order_date"], as_index=False).line_revenue.sum().rename(columns={"line_revenue": "order_value"})
orders = orders.sort_values(["customer_id", "order_date"])
g = orders.groupby("customer_id")
orders["order_seq"] = g.cumcount() + 1; orders["cum_spend"] = g.order_value.cumsum()
orders["days_since_prev"] = g.order_date.diff().dt.days; orders["pct_of_cust_total"] = orders.order_value / g.order_value.transform("sum")
orders["rank_in_cust"] = g.order_value.rank(ascending=False, method="dense")
print(orders.head(6).round({"order_value": 2, "cum_spend": 2, "pct_of_cust_total": 3})); print("median repeat gap (days):", orders.days_since_prev.median())

### 8. Time series: resample, rolling, growth

In [ ]:
h("8. Time series: resample, rolling, growth")
ts = orders.set_index("order_date").order_value.resample("MS").agg(["sum", "count"]); ts["mom_%"] = ts["sum"].pct_change() * 100
ts["ma3"] = ts["sum"].rolling(3).mean(); ts["yoy_%"] = ts["sum"].pct_change(12) * 100
print(ts.round(1).tail(6)); print("weekday revenue share:", df.groupby(df.order_date.dt.day_name()).line_revenue.sum().div(df.line_revenue.sum()).round(3).to_dict())

### 9. Feature engineering

In [ ]:
h("9. Feature engineering")
snap = df.order_date.max() + pd.Timedelta(days=1)
rfm = orders.groupby("customer_id").agg(recency=("order_date", lambda s: (snap - s.max()).days), frequency=("order_id", "nunique"),
                                        monetary=("order_value", "sum"), first=("order_date", "min"), aov=("order_value", "mean"), gap=("days_since_prev", "mean"))
rfm["tenure"] = (snap - rfm["first"]).dt.days; rfm["orders_per_month"] = rfm.frequency / (rfm.tenure / 30).clip(lower=1)
rfm["is_repeat"] = (rfm.frequency > 1).astype(int); rfm["log_monetary"] = np.log1p(rfm.monetary)
for c, asc in [("recency", False), ("frequency", True), ("monetary", True)]:
    rfm[c[0] + "_score"] = pd.qcut(rfm[c].rank(method="first", ascending=asc), 5, labels=False) + 1
rfm["RFM"] = rfm.r_score.astype(str) + rfm.f_score.astype(str) + rfm.m_score.astype(str)
print(rfm.head(3).round({"aov": 2, "gap": 1, "orders_per_month": 2, "log_monetary": 2, "monetary": 2})); print("repeat rate:", rfm.is_repeat.mean().round(3))
df["month_sin"] = np.sin(2 * np.pi * df.order_date.dt.month / 12); df["is_weekend"] = df.order_date.dt.dayofweek >= 5   # cyclical + flag
print(pd.get_dummies(df[["order_channel"]].head(3)).astype(int).to_dict("list"))

### 10. Cohort analysis

In [ ]:
h("10. Cohort analysis")
orders["cohort"] = orders.groupby("customer_id").order_date.transform("min").dt.to_period("M")
orders["idx"] = (orders.order_date.dt.to_period("M") - orders.cohort).apply(lambda p: p.n)
coh = orders.pivot_table(index="cohort", columns="idx", values="customer_id", aggfunc="nunique")
ret = coh.div(coh[0], axis=0); print(ret.iloc[:5, :7].round(2))

### 11. Seaborn gallery

In [ ]:
h("11. Seaborn gallery")
fig, ax = plt.subplots(2, 3, figsize=(17, 9))
sns.histplot(np.log1p(rfm.monetary), kde=True, ax=ax[0, 0]).set(title="log monetary + KDE")
sns.boxplot(data=df, x="acquisition_channel", y="line_revenue", ax=ax[0, 1], showfliers=False).set(title="Line revenue by channel")
sns.violinplot(data=df.sample(5000, random_state=1), x="order_channel", y="unit_price", ax=ax[0, 2]).set(title="Price by order channel")
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax[1, 0]).set(title="Correlation")
sns.scatterplot(data=rfm.sample(1500, random_state=1), x="frequency", y="monetary", hue="r_score", palette="viridis", ax=ax[1, 1]).set(title="Frequency vs monetary")
sns.heatmap(ret.iloc[:, :12], cmap="YlGnBu", ax=ax[1, 2], cbar=False).set(title="Cohort retention")
fig.tight_layout(); fig.savefig(FIG / "eda_intermediate.png", dpi=110); plt.show(); print("saved eda_intermediate.png")

### 12. Memory / dtype optimisation

In [ ]:
h("12. Memory / dtype optimisation")
before = df.memory_usage(deep=True).sum() / 1e6
opt = df.copy()
for c in opt.select_dtypes(include=["object", "string"]): opt[c] = opt[c].astype("category")
for c in opt.select_dtypes("int64"): opt[c] = pd.to_numeric(opt[c], downcast="integer")
print(f"{before:.1f} MB -> {opt.memory_usage(deep=True).sum()/1e6:.1f} MB")
rfm.reset_index().to_csv(P / "eda_rfm_features.csv", index=False)